In [100]:
import json
import os
import re
from pathlib import Path

import pandas as pd

make dict of the properties that are returned from QFP

In [76]:
with open("../data/properties.json") as f:
    properties_list = json.load(f)

property_dict = {prop["property_db_id"]: prop["name_of_property"] for prop in properties_list}
property_dict

{0: 'energy',
 1: 'atomization_energy',
 2: 'homo_lumo_gap',
 3: 'ionization_energy',
 4: 'electron_affinity',
 5: 'chemical_potential',
 6: 'molecular_dipole',
 7: 'molecular_dipole_norm',
 8: 'molecular_quadrupole',
 9: 'molecular_quadrupole_principal_invariant_2',
 10: 'molecular_quadrupole_principal_invariant_3',
 11: 'molecular_polarizability',
 12: 'molecular_polarizability_mean',
 13: 'molecular_polarizability_anisotropy',
 14: 'normal_modes',
 15: 'normal_mode_frequencies',
 16: 'infrared_intensity',
 17: 'enthalpy',
 18: 'gibbs_free_energy',
 19: 'heat_capacity',
 20: 'entropy',
 21: 'zero_point_energy',
 22: 'radius_of_gyration',
 23: 'molecular_volume',
 24: 'molecular_sasa',
 25: 'atomic_sasa',
 26: 'effective_coordination_number',
 27: 'partial_charge',
 28: 'atomic_fukui_minus',
 29: 'atomic_fukui_plus',
 30: 'atomic_dipole',
 31: 'atomic_dipole_norm',
 32: 'atomic_quadrupole',
 33: 'atomic_quadrupole_principal_invariant_2',
 34: 'atomic_quadrupole_principal_invariant_3',

The output files of the QFP program are given in the following format:

"example_input_{id}.json" provides a **list** of conformers. For each conformer, the xyz coordinates + bunch of properties

In [ ]:
def create_molecule_df(molecule_data: list[dict], property_dict: dict[int, str]) -> pd.DataFrame:
    """Transform the data from QuantumFP into a pandas dataframe. The dataframe consists of all QM properties extracted from QFP for each conformer of the molecule.

    params:
        molecule_data: A list containing a dict for each conformer of the molecule with its corresponding QM properties.

    Return:
        a pd.DataFrame containing the QM properties of each conformer of the molecule.
    """
    pattern = re.compile(r"prop_id")

    molecule_dict = {}
    for idx, conformer in enumerate(molecule_data):
        # Get all QM properties of a conformer and assign the proper name of the property
        conformer_dict = {property_dict[int(k.split("_")[-1])]: v for k, v in conformer.items() if pattern.search(k)}

        # Add the SMILES representations to the dataframe
        conformer_dict.update(
            {"original_smiles": conformer["original_smiles"], "output_smiles": conformer["output_smiles"]}
        )

        # Add the dict of the conformer to the dict of the molecule
        molecule_dict[f"conformer {idx}"] = conformer_dict

    return pd.DataFrame(molecule_dict).T

In [ ]:
def load_molecule_data(path: Path) -> list[dict]:
    with open(path) as f:
        data = json.load(f)

    return data

In [ ]:
def process_complete_output(directory: Path) -> list[pd.DataFrame]:
    # get QFP directory path
    output_files: list[Path] = os.listdir()

    # for each file, load the file and convert it to a df
    dataframes: list[pd.DataFrame] = []
    for output_file in output_files:
        output_data = load_molecule_data(output_file)

        dataframes.append(create_molecule_df(output_data))

    return dataframes

In [101]:
PATH = Path("../data/example_input_2.json")

molecule_data = load_molecule_data(PATH)
print(len(molecule_data))

29


In [102]:
molecule_df = create_molecule_df(molecule_data, property_dict)

In [105]:
molecule_df.head()

,energy,atomization_energy,homo_lumo_gap,ionization_energy,electron_affinity,chemical_potential,molecular_dipole,molecular_dipole_norm,molecular_quadrupole,molecular_quadrupole_principal_invariant_2,...,bond_energy,bond_length,bond_stiffness,overlap_integral,nuclear_repulsion,atomic_charge_dipole_interaction,atomic_charge_quadrupole_interaction,atomic_dipole_dipole_interaction,original_smiles,output_smiles
conformer 0,-85.593212,12.62387,0.055105,0.431226,0.226806,-0.330413,"[-2.013843067639362, 0.3040879389863501, -0.45...",2.087114,"[[-4.310597774234889, 18.760019952444907, -10....",-867.727217,...,"[[4, 32, 0.22197192520232534], [6, 33, 0.21103...","[[1, 2, 2.3032148036289697], [2, 3, 2.79961942...","[[1, 2, 1.4483258006561726], [2, 3, 0.84916383...","[[1, 2, 0.39282525710591715], [1, 3, 0.0426535...","[[1, 2, 11.028284427478932], [1, 3, 5.70579807...","[[1, 2, -0.0029660605554249525], [1, 3, -0.004...","[[1, 2, 0.06275366293798537], [1, 3, -8.106641...","[[2, 1, 0.002382574827838625], [3, 1, 0.001724...",[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...,[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...
conformer 1,-85.593089,12.623746,0.055001,0.431559,0.22697,-0.330799,"[-1.635170313054091, 0.0751426134147562, 0.780...",1.813453,"[[8.344310855430408, -5.231666115008593, 24.61...",-931.916356,...,"[[4, 32, 0.2250578287615923], [6, 33, 0.208882...","[[1, 2, 2.3024787455896614], [2, 3, 2.79946566...","[[1, 2, 1.4510034438462123], [2, 3, 0.85070211...","[[1, 2, 0.3996224132429931], [1, 3, 0.04622857...","[[1, 2, 11.031809957270758], [1, 3, 5.70528027...","[[1, 2, -0.0025735659297974585], [1, 3, -0.003...","[[1, 2, 0.06325213851621009], [1, 3, -0.000551...","[[2, 1, 0.0021033668741559162], [3, 1, 0.00161...",[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...,[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...
conformer 2,-85.593462,12.624119,0.059612,0.432372,0.221434,-0.327946,"[0.3371793112094228, -1.3155886903833947, -1.5...",2.055018,"[[-33.41516516359928, -29.040861996514597, -17...",-2022.077706,...,"[[4, 32, 0.22238270720885112], [6, 33, 0.21067...","[[1, 2, 2.3000906944823356], [2, 3, 2.79548502...","[[1, 2, 1.4584138533495934], [2, 3, 0.85529750...","[[1, 2, 0.33241326434030927], [1, 3, 0.0507336...","[[1, 2, 11.043263647356744], [1, 3, 5.71015415...","[[1, 2, -0.003081643159081485], [1, 3, -0.0047...","[[1, 2, 0.06350390528108867], [1, 3, -0.000139...","[[2, 1, 0.0024609717303010707], [3, 1, 0.00166...",[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...,[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...
conformer 3,-85.596029,12.626686,0.059946,0.430908,0.226144,-0.331667,"[-1.8314928322417288, -0.6112459658374106, -0....",1.936697,"[[-9.360278806243052, -14.253629229930764, -14...",-676.908159,...,"[[4, 32, 0.2223388129916941], [6, 33, 0.210990...","[[1, 2, 2.3016662149591043], [2, 3, 2.80663875...","[[1, 2, 1.4570079608014366], [2, 3, 0.84751030...","[[1, 2, 0.3693037156731059], [1, 3, 0.03987560...","[[1, 2, 11.03570438967912], [1, 3, 5.707471644...","[[1, 2, -0.0029803626784771563], [1, 3, -0.004...","[[1, 2, 0.06304432147327886], [1, 3, -0.001452...","[[2, 1, 0.0024041457637370683], [3, 1, 0.00179...",[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...,[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...
conformer 4,-85.595497,12.626154,0.060449,0.431269,0.225927,-0.331911,"[-1.4828514332353935, -1.2543510947197296, 0.8...",2.117785,"[[-3.209327417199868, -27.484131512847952, -0....",-828.759905,...,"[[4, 32, 0.22499026078540396], [6, 33, 0.21194...","[[1, 2, 2.300124121340255], [2, 3, 2.808873058...","[[1, 2, 1.4643499123746175], [2, 3, 0.84884541...","[[1, 2, 0.39901842117325664], [1, 3, 0.0424513...","[[1, 2, 11.043103159667636], [1, 3, 5.70228247...","[[1, 2, -0.0024906805684370027], [1, 3, -0.004...","[[1, 2, 0.06371047961709417], [1, 3, -0.002160...","[[2, 1, 0.0020470018047478913], [3, 1, 0.00174...",[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...,[O:1]=[C:2]([c:3]1[c:4]([H:32])[c:5]2[c:6]([H:...


Now, after processing all molecules, we need to do a thermal averaging over the different conformers (microstates) to represent the mixture of conformers that is present in reality. 

$$
<A> = \frac{\sum_i A_i e^{-E_i \beta}}{\sum_i e^{-E_i \beta}}
$$